In [0]:
# Databricks notebook source
# =============================================================================
# SILVER LAYER : f_SalesLine Transformation (UNITY CATALOG PROD STANDARD)
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, FloatType, TimestampType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import logging

# ──────────────────────────────────────────────────────────────────────────────
# 1. ENVIRONMENT CONFIGURATION & OPTIMIZATIONS
# ──────────────────────────────────────────────────────────────────────────────
# Kill the small file problem automatically!
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoOptimize.autoCompact.enabled", "true")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_f_salesline")
print("⚪ INITIALIZING PROD SILVER PIPELINE : SALES ORDER LINES")

CATALOG_NAME = "prx"
BRONZE_SCHEMA = "prx_bronze"
SILVER_SCHEMA = "prx_silver"

# Fully Qualified Unity Catalog Table Names
BRONZE_LINES_TABLE   = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.bronze_salesorderlines"
BRONZE_HEADERS_TABLE = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.bronze_salesorderheader"
SILVER_ORDER_TABLE   = f"{CATALOG_NAME}.{SILVER_SCHEMA}.salesorder"

SILVER_TABLE         = f"{CATALOG_NAME}.{SILVER_SCHEMA}.f_salesline"
EXCEPTION_TABLE      = f"{CATALOG_NAME}.{SILVER_SCHEMA}.central_exceptions"

# External Paths for initial creation
STORAGE_ACCOUNT = "stpraxaslakehouse"
SILVER_PATH            = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/f_salesline"
EXCEPTION_PATH         = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/central_exceptions"

MERGE_KEYS = ["SOSrcId", "SOLnSrcId"]

# Ensure Silver Schema Exists Securely
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")

# Capture true executing user for Auditing
executing_user = spark.sql("SELECT current_user()").collect()[0][0]

# ──────────────────────────────────────────────────────────────────────────────
# 2. UTILITIES
# ──────────────────────────────────────────────────────────────────────────────
def safe_str(c): 
    return F.coalesce(F.col(c).cast(StringType()), F.lit(""))

def safe_float(c): 
    return F.coalesce(F.col(c).cast(FloatType()), F.lit(0.0))

def empty_str_to_null_ts(c):
    return F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), None)\
            .otherwise(F.col(c)).cast(TimestampType())

# ──────────────────────────────────────────────────────────────────────────────
# 3. READ SOURCE & DIMENSIONS (BROADCAST JOINS)
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Reading Source & Dimension Data from Unity Catalog...")

# Base Source
bronze_df = spark.table(BRONZE_LINES_TABLE).alias("a")

# Lookups (Deduplicated and Broadcasted)
header_df = spark.table(BRONZE_HEADERS_TABLE).dropDuplicates(["OrderID"]).alias("c")
order_df  = spark.table(SILVER_ORDER_TABLE).dropDuplicates(["SOSrcID"]).alias("e")

logger.info("Applying Broadcast Joins...")
joined_df = bronze_df \
    .join(F.broadcast(header_df), F.col("a.OrderID") == F.col("c.OrderID"), "left") \
    .join(F.broadcast(order_df), F.col("a.OrderID") == F.col("e.SOSrcID"), "inner")

# ──────────────────────────────────────────────────────────────────────────────
# 4. CALCULATE COMPLEX EXPRESSIONS
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Applying Business & Post-Transformation Logic...")

# -- Status Calculations --
calc_soln_status = F.when((F.col("e.SOStatus") == 'Completed') & (safe_float("a.QuantityInvoiced") != 0), "Completed") \
                    .when((F.col("e.SOStatus") == 'Open') & ((safe_float("a.Quantity") == safe_float("a.QuantityDelivered")) | 
                                                             (safe_float("a.Quantity") == safe_float("a.QuantityInvoiced"))), "Completed") \
                    .when(F.col("e.SOStatus") == 'Open', "Open") \
                    .when(F.col("e.SOStatus") == 'Cancelled', "Cancelled") \
                    .when((F.col("e.SOStatus") == 'Completed') & (safe_float("a.QuantityInvoiced") == 0), "Cancelled") \
                    .otherwise("")

# -- Unit Cost Base Logic --
base_cost = F.when(safe_float("a.NetPrice") == 0, F.lit(0.0)) \
             .when(safe_float("a.CostPriceFC") == 0, F.lit(0.0)) \
             .otherwise(safe_float("a.CostPriceFC"))

# -- Quantity Logic --
calc_canc_qty = F.when(F.col("a.StatusDescription") == 'Cancelled', safe_float("a.Quantity")) \
                 .when(F.col("a.StatusDescription") == 'Completed', safe_float("a.Quantity") - safe_float("a.QuantityInvoiced")) \
                 .otherwise(0.0)

calc_rvsd_qty = F.when(F.col("a.InvoiceStatusDescription").isin('Cancelled','Completed'), safe_float("a.QuantityInvoiced")) \
                 .otherwise(safe_float("a.Quantity"))

calc_ship_qty = F.when(F.col("a.InvoiceStatusDescription") == 'Completed', safe_float("a.QuantityInvoiced"))\
                 .otherwise(safe_float("a.QuantityDelivered"))

calc_backlog_qty = F.when(calc_soln_status == 'Open', calc_rvsd_qty - calc_ship_qty).otherwise(0.0)

# -- Pricing & Multipliers --
exrt_cclc = safe_float("e.ExRtCCLC")
exrt_lcus = safe_float("e.ExRtLCUS")

lat_unit_cost_cc = base_cost
lat_unit_price_cc = safe_float("a.UnitPrice")
disc_perc = safe_float("c.Discount")
disc_unit_price_cc = safe_float("a.NetPrice") - (disc_perc * safe_float("a.NetPrice"))

is_rma = F.col("e.SOTrxnType") == 'RMA'

# ──────────────────────────────────────────────────────────────────────────────
# 5. BUILD THE SILVER DATAFRAME
# ──────────────────────────────────────────────────────────────────────────────
silver_df = joined_df.select(
    # Identifiers
    safe_str("a.OrderID").alias("SOSrcId"),
    safe_str("a.ID").alias("SOLnSrcId"),
    safe_str("a.PurchaseOrder").alias("POSrcID"),
    safe_str("a.PurchaseOrderLine").alias("POLnSrcID"),
    F.lit("").alias("VendSrcId"),
    safe_str("c.WarehouseID").alias("WHSrcId"),
    F.lit("").alias("ProductLnSrcId"),
    safe_str("a.Item").alias("ItemSrcId"),
    F.lit("").alias("GLAcctKey"),
    F.lit("").alias("JrnlKey"),
    safe_str("a.OrderNumber").alias("SOKey"),
    safe_str("a.LineNumber").alias("SOLnKey"),
    F.lit(0).alias("SOLnSeqKey"),
    F.lit("").alias("POKey"),
    F.lit("").alias("POLnKey"),
    
    # Statuses
    calc_soln_status.alias("SOLnStatus"),
    F.when((F.col("a.DeliveryStatusDescription") == 'Complete') & (safe_float("a.QuantityInvoiced") != 0), "FullyShipped") \
     .when(F.col("a.DeliveryStatusDescription").isin('Open','Partial') & (safe_float("a.QuantityDelivered") != 0), "PartiallyShipped") \
     .when(F.col("a.DeliveryStatusDescription").isin('Open','Partial') & (safe_float("a.QuantityDelivered") == 0), "Yet To Ship") \
     .when(F.col("a.DeliveryStatusDescription") == 'Cancelled', "Cancelled") \
     .when((F.col("a.DeliveryStatusDescription") == 'Complete') & (safe_float("a.QuantityInvoiced") == 0), "Cancelled") \
     .otherwise("").alias("SOLnShipStatus"),
     
    F.when((F.col("a.InvoiceStatusDescription") == 'Complete') & (safe_float("a.QuantityInvoiced") != 0), "Completed") \
     .when(F.col("a.InvoiceStatusDescription").isin('Open','Partial'), "Open") \
     .when(F.col("a.InvoiceStatusDescription") == 'Cancelled', "Cancelled") \
     .when((F.col("a.InvoiceStatusDescription") == 'Complete') & (safe_float("a.QuantityInvoiced") == 0), "Cancelled") \
     .otherwise("").alias("InvLnStatus"),
     
    # Categories & Types
    safe_str("e.SOOrdCat").alias("SOLnCat"),
    safe_str("e.SOOrdCatDesc").alias("SOLnCatDesc"),
    F.lit("SO").alias("SOLnTrxnType"),
    F.lit("SalesOrder").alias("SOLnTrxnTypeDesc"),
    safe_str("e.SOTrxnType").alias("SOLnType"),
    safe_str("e.SOTrxnTypeDesc").alias("SOLnTypeDesc"),
    
    # Dates
    empty_str_to_null_ts("a.DeliveryDate").alias("PlanShipDt"),
    empty_str_to_null_ts("a.DeliveryDate").alias("PromiseDt"),
    F.when(calc_soln_status == 'Complete', empty_str_to_null_ts("e.ShipDt")).otherwise(F.lit(None)).cast(TimestampType()).alias("ActlShipDt"),
    F.when(calc_soln_status == 'Cancelled', empty_str_to_null_ts("a.Modified")).otherwise(F.lit(None)).cast(TimestampType()).alias("CancDt"),

    # Product & Warehouse
    safe_str("c.WarehouseCode").alias("WHKey"),
    safe_str("c.WarehouseDescription").alias("WHName"),
    safe_str("a.ItemCode").alias("ItemKey"),
    F.lit("").alias("ItemType"),
    safe_str("a.ItemDescription").alias("ItemDesc"),
    safe_str("a.Description").alias("ItemAddlDesc1"),
    safe_str("a.UnitCode").alias("UOM"),
    safe_str("a.UnitDescription").alias("UOMDesc"),
    safe_str("a.CostCenter").alias("SOLnCostType"),
    safe_str("a.CostCenterDescription").alias("SOLnCostTypeDesc"),
    
    # Currency
    safe_str("e.CustCurr").alias("CustCurr"),
    F.lit("EUR").alias("LocCurr"),
    safe_float("e.ExDt").alias("ExDt"),
    safe_float("e.ExRtLCCC").alias("ExRtLCCC"),
    exrt_cclc.alias("ExRtCCLC"),
    exrt_lcus.alias("ExRtLCUS"),

    # Quantities
    safe_float("a.Quantity").alias("SOLnOrgQty"),
    safe_float("a.Quantity").alias("SOLnOrdQty"),
    calc_canc_qty.alias("SOLnCancQty"),
    calc_rvsd_qty.alias("SOLnRvsdQty"),
    calc_ship_qty.alias("ShippedQty"),
    calc_backlog_qty.alias("BacklogQty"),
    safe_float("a.QuantityInvoiced").alias("InvLnQty"),
    safe_float("a.QuantityDelivered").alias("DlvryQty"),

    # Costs
    base_cost.alias("IniUnitCostCC"),
    (base_cost * exrt_cclc).alias("IniUnitCostLC"),
    (base_cost * exrt_cclc * exrt_lcus).alias("IniUnitCostUS"),
    lat_unit_cost_cc.alias("LatUnitCostCC"),
    (lat_unit_cost_cc * exrt_cclc).alias("LatUnitCostLC"),
    (lat_unit_cost_cc * exrt_cclc * exrt_lcus).alias("LatUnitCostUS"),

    # Prices
    lat_unit_price_cc.alias("IniUnitPriceCC"),
    (lat_unit_price_cc * exrt_cclc).alias("IniUnitPriceLC"),
    (lat_unit_price_cc * exrt_cclc * exrt_lcus).alias("IniUnitPriceUS"),
    lat_unit_price_cc.alias("LatUnitPriceCC"),
    (lat_unit_price_cc * exrt_cclc).alias("LatUnitPriceLC"),
    (lat_unit_price_cc * exrt_cclc * exrt_lcus).alias("LatUnitPriceUS"),

    disc_perc.alias("SODiscPerc"),
    safe_float("a.Discount").alias("DiscPerc"),
    disc_unit_price_cc.alias("DiscUnitPriceCC"),
    (disc_unit_price_cc * exrt_cclc).alias("DiscUnitPriceLC"),
    (disc_unit_price_cc * exrt_cclc * exrt_lcus).alias("DiscUnitPriceUS"),

    # Post-Transform Valuations
    (calc_rvsd_qty * disc_unit_price_cc).alias("SOLnValCC"),
    (calc_rvsd_qty * (disc_unit_price_cc * exrt_cclc)).alias("SOLnValLC"),
    (calc_rvsd_qty * (disc_unit_price_cc * exrt_cclc * exrt_lcus)).alias("SOLnValUS"),
    
    (calc_rvsd_qty * (lat_unit_price_cc - disc_unit_price_cc)).alias("SOLnDiscValCC"),
    (calc_rvsd_qty * ((lat_unit_price_cc * exrt_cclc) - (disc_unit_price_cc * exrt_cclc))).alias("SOLnDiscValLC"),
    (calc_rvsd_qty * ((lat_unit_price_cc * exrt_cclc * exrt_lcus) - (disc_unit_price_cc * exrt_cclc * exrt_lcus))).alias("SOLnDiscValUS"),

    (calc_rvsd_qty * lat_unit_cost_cc).alias("SOLnCostCC"),
    (calc_rvsd_qty * (lat_unit_cost_cc * exrt_cclc)).alias("SOLnCostLC"),
    (calc_rvsd_qty * (lat_unit_cost_cc * exrt_cclc * exrt_lcus)).alias("SOLnCostUS"),

    # Taxes (Applying RMA Tax Nullification Logic natively)
    F.when(~is_rma, safe_float("a.VATAmount") - (disc_perc * safe_float("a.VATAmount"))).otherwise(0.0).alias("SOLnTaxValCC"),
    F.when(~is_rma, (safe_float("a.VATAmount") - (disc_perc * safe_float("a.VATAmount"))) * exrt_cclc).otherwise(0.0).alias("SOLnTaxValLC"),
    F.when(~is_rma, (safe_float("a.VATAmount") - (disc_perc * safe_float("a.VATAmount"))) * exrt_cclc * exrt_lcus).otherwise(0.0).alias("SOLnTaxValUS"),

    safe_str("a.VATCode").alias("TaxCd"),
    safe_str("a.VATCodeDescription").alias("TaxCdDesc"),
    safe_str("a.Pricelist").alias("PriceListCd"),
    safe_str("a.PricelistDescription").alias("PriceListDesc"),
    safe_str("a.Notes").alias("AddInfo"),
    
    # Flags
    F.coalesce(F.col("a.UseDropShipment"), F.lit("No")).alias("IsDropShipFl"),
    F.lit("No").alias("IsDeletedFl"),
    F.lit("SalesLine").alias("SrcTable"),

    # Source Audit & Enterprise Pipeline Audit
    empty_str_to_null_ts("a.Created").alias("CreatedDt"),
    safe_str("a.CreatorFullName").alias("CreatedBy"),
    empty_str_to_null_ts("a.Modified").alias("UpdatedDt"),
    safe_str("a.ModifierFullName").alias("UpdatedBy"),
    F.current_timestamp().alias("SysUpdatedDt"),
    F.current_timestamp().alias("SysCreatedDt"),
    F.lit(executing_user).alias("SysUpdatedBy")
)

# ──────────────────────────────────────────────────────────────────────────────
# 6. DATA QUALITY & DEDUPLICATION
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Evaluating Data Quality Rules...")
dq_df = silver_df.withColumn(
    "DQ_Reason",
    F.array_remove(F.array(
        F.when(F.col("SOSrcId") == "", F.lit("MISSING_SO_ID")),
        F.when(F.col("SOLnSrcId") == "", F.lit("MISSING_LN_ID"))
    ), None)
).withColumn(
    "DQ_Status",
    F.when(F.size("DQ_Reason") > 0, F.lit(2)).otherwise(F.lit(0))
)

window_dup = Window.partitionBy("SOSrcId", "SOLnSrcId").orderBy(F.col("UpdatedDt").desc_nulls_last())

dq_df = dq_df.withColumn(
    "rn", F.row_number().over(window_dup)
).withColumn(
    "DQ_Status",
    F.when(F.col("rn") > 1, F.lit(4)).otherwise(F.col("DQ_Status"))
).drop("rn")

# Cache to prevent massive DAG re-computes!
dq_df.cache()

valid_df = dq_df.filter(F.col("DQ_Status") == 0).drop("DQ_Reason", "DQ_Status")
error_df = dq_df.filter(F.col("DQ_Status").isin(2, 4))

valid_count = valid_df.count()
error_count = error_df.count()

logger.info(f"✅ Valid Records: {valid_count:,}")
logger.info(f"❌ Error Records: {error_count:,}")

# ──────────────────────────────────────────────────────────────────────────────
# 7. CENTRAL EXCEPTION ROUTING
# ──────────────────────────────────────────────────────────────────────────────
if error_count > 0:
    logger.info("Routing exceptions to Unity Catalog Quarantine...")
    exception_df = error_df.select(
        F.lit(SILVER_TABLE).alias("table_name"),
        F.concat_ws("-", F.col("SOSrcId"), F.col("SOLnSrcId")).alias("record_key"), 
        F.when(F.col("DQ_Status") == 4, F.lit("Duplicate Value"))
         .otherwise(F.concat(F.lit("Validation Failed: "), F.concat_ws(", ", F.col("DQ_Reason")))).alias("exception_details"),
        F.current_timestamp().alias("syscreateddt")
    )

    if spark.catalog.tableExists(EXCEPTION_TABLE):
        exception_df.write.format("delta").mode("append").saveAsTable(EXCEPTION_TABLE)
    else:
        exception_df.write.format("delta").mode("overwrite").option("path", EXCEPTION_PATH).saveAsTable(EXCEPTION_TABLE)

# ──────────────────────────────────────────────────────────────────────────────
# 8. MERGE INTO SILVER DELTA
# ──────────────────────────────────────────────────────────────────────────────
if valid_count > 0:
    logger.info("Executing MERGE INTO Silver Delta Table...")
    window_spec = Window.partitionBy(*MERGE_KEYS).orderBy(F.col("UpdatedDt").desc_nulls_last())
    final_df = valid_df.withColumn("rn", F.row_number().over(window_spec)).filter("rn = 1").drop("rn")

    if spark.catalog.tableExists(SILVER_TABLE):
        delta_tbl = DeltaTable.forName(spark, SILVER_TABLE)
        cond = " AND ".join([f"tgt.{k} = src.{k}" for k in MERGE_KEYS])
        
        (delta_tbl.alias("tgt")
         .merge(final_df.alias("src"), cond)
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        final_df.write.format("delta").mode("overwrite").option("path", SILVER_PATH).saveAsTable(SILVER_TABLE)

dq_df.unpersist()
print("🎉 PROD SALES LINES PIPELINE COMPLETED SUCCESSFULLY!")

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT * FROM prx.prx_silver.f_salesline LIMIT 10;